Cloning into 'Depi-project-'...
remote: Enumerating objects: 122, done.
remote: Counting objects: 100% (122/122), done.
remote: Compressing objects: 100% (96/96), done.
remote: Total 122 (delta 58), reused 58 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (122/122), 45.11 MiB | 16.16 MiB/s, done.
Resolving deltas: 100% (58/58), done.
/content/Depi-project-
  Preparing metadata (setup.py) ... done


In [2]:
!python run_pipeline.py --excel All_train_data.xlsx \
    --epochs 100 \
    --hidden-dim 128 \
    --n-lstm-layers 3 \
    --window-size 50 \
    --lr 0.001 \
    --batch-size 128 \
    --early-stop-patience 15 \
    --export-preds predictions.csv

Step 1 - Load all sheets from All_train_data.xlsx
  Loaded sheet 'train_FD001(HPC Degradation)': 100 engines, 20631 rows
  Loaded sheet 'train_FD002(HPC Degradation)': 260 engines, 53759 rows
  Loaded sheet 'train_FD003(HPC+Fan_Deg)': 100 engines, 24720 rows
  Loaded sheet 'train_FD004(HPC+Fan_Deg)': 249 engines, 61249 rows
  Total: 709 engines, 160359 rows
  Shape   : (160359, 27)
  Engines : 709
  Columns : ['unit_id', 'cycle', 'op_set_1', 'op_set_2', 'op_set_3', 's1', 's2', 's3'] ... (27 total)

Step 2 - Validate & clean (impute, drop flat sensors)
  Cleaned shape : (160359, 26)
  Removed flat  : ['s16']

Step 3 - Compute RUL + classification labels
  RUL range     : [0, 125]
  Label balance : {0: 138380, 1: 21979}

Step 4 - Build sliding windows (no future leakage)
  X shape       : (125618, 50, 23)  (samples, window, features)
  y_cls shape   : (125618,)
  y_rul shape   : (125618,)
  Engines used  : 709

Step 5 - Extract engineered features per window
  Engineered shape : (125618,

Got you 👍 — here’s the same content cleanly formatted in **professional English tables**:

---

## 🔹 1. Model Bias: Under-Predicts RUL (Conservative)

| Metric            | Value                                |
| ----------------- | ------------------------------------ |
| Mean Error        | **-1.65** (negative = under-predict) |
| Under-predictions | **61% of samples**                   |
| Over-predictions  | **39% of samples**                   |

🧠 **Insight:**
The model is conservative, which is beneficial for safety since it predicts failure earlier than it actually occurs.

---

## 🔹 2. Error Distribution

| Metric                    | Value            |
| ------------------------- | ---------------- |
| Mean Absolute Error (MAE) | **10.37 cycles** |
| Median Absolute Error     | **6.17 cycles**  |
| Max Error                 | **120 cycles**   |

📊 **Insight:**
50% of predictions fall within ±6 cycles of the true RUL, indicating solid overall accuracy.

---

## 🔹 3. Worst Predictions (High RUL Issue)

| Attribute     | Observation                          |
| ------------- | ------------------------------------ |
| True RUL      | **125 (maximum capped value)**       |
| Predicted RUL | **5 – 15 (severe under-prediction)** |
| Engine IDs    | **14519 – 14538**                    |
| Dataset       | **Likely FD004 (most complex)**      |

⚠️ **Explanation:**

* These engines are at the **start of their lifecycle** (very healthy).
* The model struggles to distinguish between:

  * New engines
  * Early-stage degradation

📌 **Root Causes:**

* Weak degradation signals at high RUL
* RUL capping at 125 leads to label saturation

---

## 🔹 4. Error by RUL Range

| RUL Range             | Mean Error | Interpretation             |
| --------------------- | ---------- | -------------------------- |
| 0 – 30 (near failure) | **+2.45**  | Slight over-prediction ✅   |
| 30 – 60               | **+4.32**  | Over-prediction ✅          |
| 60 – 90               | **+6.86**  | Over-prediction ✅          |
| 90 – 125 (healthy)    | **-6.88**  | Under-prediction ❌ (issue) |

🎯 **Key Insight:**

* Strong performance in **near-failure prediction**
* Weak performance for **healthy engines (high RUL)**

---

## Summary

* ✅ Safe and conservative model behavior
* ✅ Strong performance near failure
* ❌ Weak performance for early-life / healthy engines
* ⚠️ RUL capping introduces noticeable bias

---
